# 02 · Exploratory data analysis

Descriptive analysis of the **real** Ausgrid + ERA5 tensors — the figures that go in the paper's
Data section. Nothing here is simulated; every plot is drawn from the aligned arrays built in
notebook 01.

Figures produced:
1. Aggregate demand & generation over the full 2010–2013 span (train/test shaded).
2. Mean daily profiles by season (demand and PV) — the diurnal + seasonal structure the model must learn.
3. Distribution of demand and generation (note the night-time PV mass at zero → motivates the day-mask metric).
4. Cross-node correlation of hourly demand (justifies the spatial graph).
5. Graph degree distribution and edge-weight histogram.
6. Weather–demand / weather–PV correlation heatmap.

In [ ]:
import os, sys, pickle
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, os.path.abspath('..'))
from src.utils import load_config
from src.data.dataset import WEATHER_COLS, DEMAND_CH, GEN_CH, calendar_features

cfg = load_config('../configs/default.yaml')
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})

with open(Path(cfg['data']['processed_dir']) / 'dataset.pkl', 'rb') as f:
    dm = pickle.load(f)
print('loaded data module: nodes=%d, train=%d, test=%d' % (
    dm.train_ds.n_nodes, len(dm.train_ds), len(dm.test_ds)))

## 1 · Aggregate demand & generation over time

Read the raw kWh arrays (`raw_demand`, `raw_gen`) — never the scaled tensor — for interpretability.

In [ ]:
al = dm.aligned
master = pd.to_datetime(al['master_index'])
raw_d = al['raw_demand']; raw_g = al['raw_gen']   # (N, T) kWh

agg_d = raw_d.mean(0)
agg_g = raw_g.mean(0)
train_end = pd.to_datetime(cfg['data']['train_end'])

fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(master, agg_d, lw=0.4, color='C0'); ax[0].set_ylabel('mean demand (kWh/h)')
ax[1].plot(master, agg_g, lw=0.4, color='C1'); ax[1].set_ylabel('mean PV gen (kWh/h)')
for a in ax:
    a.axvspan(train_end, master[-1], color='grey', alpha=0.12)
    a.text(train_end, a.get_ylim()[1]*0.8, ' test', color='grey')
ax[0].set_title('Aggregate hourly demand & generation (all nodes) — real Ausgrid data')
plt.tight_layout(); plt.savefig('fig_aggregate.png', bbox_inches='tight'); plt.show()

## 2 · Mean daily profiles by season

Seasonal structure is what the calendar channels + lookback window must exploit; the PV curve
collapsing to zero at night is why the evaluation reports a day-masked generation metric.

In [ ]:
hour = master.hour
month = master.month
season = pd.cut(month, bins=[0,2,5,8,11,12],
                labels=['summer','autumn','winter','spring','summer2'], ordered=False)
season = season.astype(str).str.replace('summer2','summer')

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for s in ['summer','autumn','winter','spring']:
    m = (season == s).values
    prof_d = pd.Series(agg_d[m]).groupby(hour[m]).mean()
    prof_g = pd.Series(agg_g[m]).groupby(hour[m]).mean()
    ax[0].plot(prof_d.index, prof_d.values, label=s)
    ax[1].plot(prof_g.index, prof_g.values, label=s)
ax[0].set_title('Mean daily demand profile by season'); ax[0].set_xlabel('hour'); ax[0].set_ylabel('kWh/h')
ax[1].set_title('Mean daily PV profile by season'); ax[1].set_xlabel('hour')
ax[0].legend(); ax[1].legend()
plt.tight_layout(); plt.savefig('fig_daily_profiles.png', bbox_inches='tight'); plt.show()

## 3 · Distributions

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(raw_d.reshape(-1), bins=80, color='C0'); ax[0].set_title('Demand (kWh/h)'); ax[0].set_yscale('log')
ax[1].hist(raw_g.reshape(-1), bins=80, color='C1'); ax[1].set_title('PV generation (kWh/h)'); ax[1].set_yscale('log')
for a in ax: a.set_ylabel('count (log)')
frac_night_zero = float((raw_g == 0).mean())
print('fraction of generation samples == 0 (night-time mass): %.3f' % frac_night_zero)
plt.tight_layout(); plt.savefig('fig_distributions.png', bbox_inches='tight'); plt.show()

## 4 · Cross-node demand correlation (justifies the spatial graph)

If nodes were independent the GAT stream would be pointless. We check the mean pairwise
correlation of hourly demand — high positive correlation supports modelling shared structure.

In [ ]:
from itertools import combinations
samp = raw_d[: min(40, raw_d.shape[0])]
cors = [np.corrcoef(samp[i], samp[j])[0,1] for i,j in combinations(range(len(samp)), 2)]
cors = np.array(cors)
print('mean pairwise demand correlation: %.3f (median %.3f)' % (np.nanmean(cors), np.nanmedian(cors)))
plt.figure(figsize=(6,3.5))
plt.hist(cors, bins=40, color='C2'); plt.xlabel('Pearson r (hourly demand)'); plt.ylabel('pairs')
plt.title('Pairwise demand correlation across nodes'); plt.tight_layout()
plt.savefig('fig_node_correlation.png', bbox_inches='tight'); plt.show()

## 5 · Graph degree & edge-weight distribution

In [ ]:
nbr = dm.graph['neighbor_ids']; wts = dm.graph['neighbor_weight']
deg = np.array([sum(1 for x in row if x >= 0) for row in nbr])
all_w = np.array([w for row in wts for w in row if w > 0])
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(deg, bins=range(0, dm.graph['n_nodes']+2), color='C3')
ax[0].set_title('Graph degree distribution'); ax[0].set_xlabel('# neighbours (≤ K)'); ax[0].set_ylabel('nodes')
ax[1].hist(all_w, bins=40, color='C4')
ax[1].set_title('Edge-weight distribution'); ax[1].set_xlabel('cosine profile similarity')
print('mean degree: %.2f | max K: %d | mean edge weight: %.3f' % (deg.mean(), dm.train_ds.K, all_w.mean() if len(all_w) else float('nan')))
plt.tight_layout(); plt.savefig('fig_graph_stats.png', bbox_inches='tight'); plt.show()

## 6 · Weather ↔ demand / PV correlations

Expected: strong positive shortwave-radiation ↔ PV; temperature ↔ demand (cooling) and, in winter,
heating. These relationships are the justification for the weather encoder stream.

In [ ]:
# weather channels live at columns 2..10 of the scaled tensor; de-standardise for EDA
w_scaled = al['data'][0, :, 2:2+len(WEATHER_COLS)]
w = dm.weather_scaler.inverse_transform(w_scaled)
wdf = pd.DataFrame(w, columns=WEATHER_COLS)
wdf['demand'] = agg_d
wdf['gen']    = agg_g
corr = wdf.corr()[['demand','gen']].drop(index=['demand','gen'])

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks([0,1]); ax.set_xticklabels(['demand','gen']); ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.index, fontsize=8)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, shrink=0.7); ax.set_title('Weather ↔ target correlation')
plt.tight_layout(); plt.savefig('fig_weather_corr.png', bbox_inches='tight'); plt.show()

### EDA takeaways (fill in once run on real data)
- Note the actual mean pairwise demand correlation and mean graph degree above.
- Note the night-time-zero fraction for generation — it sets the day-mask threshold used in evaluation.
- These are the numbers the paper's Data section should quote; do not invent them.